# Lending Club Loan Performance & Risk Analysis (2007–2018)

Peer-to-peer lending platforms must carefully balance credit risk management with loan portfolio profitability. Ineffective risk assessment can lead to higher default rates, reduced investor confidence, and lower financial sustainability.

Lending institutions therefore need to better understand how borrower characteristics, loan attributes, and financial behavior influence loan performance and default risk.

Using historical lending data from Lending Club (2007–2018), this project analyzes loan outcomes to uncover patterns in borrower profiles, loan characteristics, and repayment behavior.

The objective is to generate data-driven insights that can help lending companies:

- Identify high-risk borrower profiles

- Understand the relationship between loan characteristics and default rates

- Evaluate the profitability of different loan segments

- Support better credit risk and lending strategy decisions

The analysis uses Python for data cleaning and exploratory analysis, SQL for business queries, and Power BI to create an interactive dashboard for financial insights.

---

## Connecting dataset

In [22]:
%load_ext sql

%sql sqlite:///../data/cleaned_data.db

%config SqlMagic.style = '_DEPRECATED_DEFAULT'

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


---

## Overview

First take a look of the clean data to have a general overview of the lending portfolio:

In [37]:
%%sql
SELECT *
FROM loans
LIMIT 5;

 * sqlite:///../data/cleaned_data.db
Done.


grade,delinq_2yrs,inq_last_6mths,dti,annual_inc,revol_util,loan_amnt,int_rate,term,issue_d,installment,earliest_cr_line,total_acc,open_acc,mths_since_last_delinq,total_pymnt,default,purpose_group,fico_score
C,0,1,5.91,55000.0,29.7,3600.0,13.99,36,2015-12-01 00:00:00.000000,123.03,2003-08-01 00:00:00.000000,13,7,30,4421.72,0,Debt,677.0
C,1,4,16.06,65000.0,19.2,24700.0,11.99,36,2015-12-01 00:00:00.000000,820.28,1999-12-01 00:00:00.000000,38,22,6,25679.66,0,Business,717.0
B,0,0,10.78,63000.0,56.2,20000.0,10.78,60,2015-12-01 00:00:00.000000,432.66,2000-08-01 00:00:00.000000,18,6,0,22705.92,0,Home,697.0
F,1,3,25.37,104433.0,64.5,10400.0,22.45,60,2015-12-01 00:00:00.000000,289.91,1998-06-01 00:00:00.000000,35,12,12,11740.5,0,Consumer,697.0
C,0,0,10.2,34000.0,68.4,11950.0,13.44,36,2015-12-01 00:00:00.000000,405.18,1987-10-01 00:00:00.000000,6,5,0,13708.95,0,Debt,692.0


---

Default rate

In [44]:
%%sql
SELECT 
    COUNT(*) AS total_loans,
    SUM("default") AS total_defaults,
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate_percent
FROM loans;

 * sqlite:///../data/cleaned_data.db
Done.


total_loans,total_defaults,default_rate_percent
1371165,294415,21.47


Default Rate by Credit Grade

In [46]:
%%sql
SELECT 
    grade,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate
FROM loans
GROUP BY grade
ORDER BY grade;

 * sqlite:///../data/cleaned_data.db
Done.


grade,loans,defaults,default_rate
A,236758,15869,6.7
B,398528,58357,14.64
C,390724,94687,24.23
D,206696,66797,32.32
E,96224,38609,40.12
F,32828,15261,46.49
G,9407,4835,51.4


Default Rate by Income Level

In [48]:
%%sql
SELECT 
    CASE
        WHEN annual_inc < 40000 THEN 'Low Income'
        WHEN annual_inc < 80000 THEN 'Middle Income'
        ELSE 'High Income'
    END AS income_group,
    
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate
    
FROM loans
GROUP BY income_group;

 * sqlite:///../data/cleaned_data.db
Done.


income_group,loans,defaults,default_rate
High Income,480682,88863,18.49
Low Income,214632,54213,25.26
Middle Income,675851,151339,22.39


Default Rate by Loan Amount

In [50]:
%%sql
SELECT 
    CASE
        WHEN loan_amnt < 5000 THEN 'Small'
        WHEN loan_amnt < 15000 THEN 'Medium'
        ELSE 'Large'
    END AS loan_size,
    
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate

FROM loans
GROUP BY loan_size;

 * sqlite:///../data/cleaned_data.db
Done.


loan_size,loans,defaults,default_rate
Large,590009,144248,24.45
Medium,646277,127803,19.78
Small,134879,22364,16.58


Average Interest Rate by Grade

In [51]:
%%sql
SELECT 
    grade,
    ROUND(AVG(int_rate),2) AS avg_interest_rate
FROM loans
GROUP BY grade
ORDER BY grade;

 * sqlite:///../data/cleaned_data.db
Done.


grade,avg_interest_rate
A,7.11
B,10.68
C,14.03
D,17.75
E,21.21
F,25.0
G,27.8


Total Interest Revenue

In [54]:
%%sql
SELECT 
    ROUND(SUM(total_pymnt - loan_amnt),2) AS total_profit
FROM loans;

 * sqlite:///../data/cleaned_data.db
Done.


total_profit
337352631.21


Profit by Grade

In [55]:
%%sql
SELECT 
    grade,
    ROUND(SUM(total_pymnt - loan_amnt),2) AS profit
FROM loans
GROUP BY grade
ORDER BY profit DESC;

 * sqlite:///../data/cleaned_data.db
Done.


grade,profit
B,223398529.7
A,159560656.3
C,73171868.86
G,-18312837.6
F,-29219974.8
D,-32053846.67
E,-39191764.58


Default Rate by FICO Score

In [57]:
%%sql
SELECT 
    CASE
        WHEN fico_score < 600 THEN 'Poor'
        WHEN fico_score < 700 THEN 'Fair'
        WHEN fico_score < 750 THEN 'Good'
        ELSE 'Excellent'
    END AS credit_group,
    
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate

FROM loans
GROUP BY credit_group;

 * sqlite:///../data/cleaned_data.db
Done.


credit_group,loans,defaults,default_rate
Excellent,108092,10964,10.14
Fair,837712,210540,25.13
Good,425361,72911,17.14
